<a href="https://colab.research.google.com/github/andrezasdias/gee/blob/main/mapeamento_wri_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [55]:
import ee
import time

# ==============================================================================
# 1. CONFIGURAÇÃO E AUTENTICAÇÃO COM PROJETO VINCULADO
# ==============================================================================
nome_do_projeto = 'atividadeifbaiano-464811'

try:
    ee.Initialize(project=nome_do_projeto)
    print("Google Earth Engine inicializado com sucesso!")
except Exception as e:
    print("Solicitando autenticação ao Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=nome_do_projeto)
    print("Autenticado e inicializado com sucesso!")

Google Earth Engine inicializado com sucesso!


In [59]:
# ==============================================================================
# 2. DEFINIÇÃO DAS VARIÁVEIS E SEUS ASSETS REAIS
# ==============================================================================
bandas = ['B2', 'B3', 'B4', 'B8']

geometria_alvo = ee.FeatureCollection('projects/atividadeifbaiano-464811/assets/grade_cerrado_50km_clipada')
amostras_locais = ee.FeatureCollection('projects/atividadeifbaiano-464811/assets/pontos_cerrado_teste_v4')

# GERAÇÃO AUTOMÁTICA DA LISTA EM LOOPING (Do ID 1 até o ID 1054)
# O range para no número anterior, por isso usamos 1055.
lista_de_quadrados = list(range(1, 1055))

print(f"🚀 Preparando envio em lote para {len(lista_de_quadrados)} quadrados.")
print("As ordens de processamento serão enviadas diretamente aos servidores do Google.\n")
print("-" * 60)

# Contadores para o relatório final do Colab
sucessos = 0
pulados = 0
erros = 0

🚀 Preparando envio em lote para 1054 quadrados.
As ordens de processamento serão enviadas diretamente aos servidores do Google.

------------------------------------------------------------


In [61]:
# ==============================================================================
# BLOCO UNIFICADO E CORRIGIDO (COLE TUDO EM UMA ÚNICA CÉLULA)
# ==============================================================================

# 1. DEFINIÇÃO DAS FUNÇÕES AUXILIARES (Fora do loop para garantir estabilidade)
def preparar_pontos(feat):
    return feat.set('classe_num', ee.Number.parse(feat.get('classe')))

def definir_zero(feat):
    return feat.set('classe_num', 0)


# 2. INÍCIO DO LOOPING EM MASSA
for id_atual in lista_de_quadrados:

    # 3. Isolar o quadrado selecionado
    quadrado_isolado = geometria_alvo.filter(ee.Filter.eq('ID', id_atual))

    # Se o quadrado físico não existir na grade, pula direto
    if quadrado_isolado.size().getInfo() == 0:
        print(f"⚠️ [ID {id_atual}]: Não encontrado na grade. Pulando...")
        pulados += 1
        continue

    # Extrai a geometria com segurança
    BlackBox_geometria = quadrado_isolado.geometry()

    # 4. MOSAICO SENTINEL-2 ULTRA OTIMIZADO (CLIP EFICIENTE)
    imagem_quadrado = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                       .filterBounds(BlackBox_geometria)
                       .filterDate('2024-01-01', '2026-05-19')
                       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
                       .median()
                       .clip(BlackBox_geometria))

    # 5. Filtrar as amostras de Pastagem dentro deste espaço
    pontos_pastagem = amostras_locais.filterBounds(BlackBox_geometria).map(preparar_pontos)

    # Proteção: Se o quadrado não tiver pontos de pastagem, pula para evitar erro no Random Forest
    qtd_pontos = pontos_pastagem.size().getInfo()
    if qtd_pontos == 0:
        print(f"ℹ️ [ID {id_atual}]: Zero pontos de pastagem encontrados. Pulando setor...")
        pulados += 1
        continue

    # 6. Gerar amostras automáticas de Não-Pastagem (Classe 0) apenas nesta área
    pontos_nao_pastagem = ee.FeatureCollection.randomPoints(
        region=BlackBox_geometria,
        points=qtd_pontos * 2,
        seed=42
    ).map(definir_zero)

    # 7. Unir as classes para o treinamento
    amostras_treinamento = pontos_pastagem.merge(pontos_nao_pastagem)

    # 8. Extrair os dados espectrais dos pixels selecionados
    dados_treinamento = imagem_quadrado.sampleRegions(
        collection=amostras_treinamento,
        properties=['classe_num'],
        scale=10,
        tileScale=16
    ).filter(ee.Filter.notNull(bandas))

    # Verificar se o GEE conseguiu extrair pixels válidos na amostragem espectral
    if dados_treinamento.size().getInfo() == 0:
        print(f"❌ [ID {id_atual}]: Sem dados espectrais válidos (nuvens/sombra). Pulando...")
        pulados += 1
        continue

    # 9. TREINAMENTO E CLASSIFICAÇÃO (RANDOM FOREST)
    classificador_local = ee.Classifier.smileRandomForest(100).train(
        features=dados_treinamento,
        classProperty='classe_num',
        inputProperties=bandas
    )

    classificacao = imagem_quadrado.select(bandas).classify(classificador_local)

    # ==============================================================================
    # 10. EXPORTAÇÃO AUTOMÁTICA DIRETO PARA O DRIVE
    # ==============================================================================
    nome_da_tarefa = f'Classificacao_Pastagem_Quadrado_{id_atual}'

    tarefa = ee.batch.Export.image.toDrive(
        image=classificacao,
        description=nome_da_tarefa,
        folder='Mapeamento_Pastagem_Cerrado_50km',
        scale=10,
        region=BlackBox_geometria,
        maxPixels=1e13,
        shardSize=256
    )

    # Inicia a tarefa em background nos servidores da Google
    tarefa.start()
    print(f"✅ [ID {id_atual}]: Adicionado com sucesso à fila de processamento da nuvem!")
    sucessos += 1

    # Pausa de segurança para não estourar o limite de requisições por minuto da API
    time.sleep(1.2)

✅ [ID 1]: Adicionado com sucesso à fila de processamento da nuvem!
✅ [ID 2]: Adicionado com sucesso à fila de processamento da nuvem!
✅ [ID 3]: Adicionado com sucesso à fila de processamento da nuvem!
✅ [ID 4]: Adicionado com sucesso à fila de processamento da nuvem!
✅ [ID 5]: Adicionado com sucesso à fila de processamento da nuvem!


KeyboardInterrupt: 

In [58]:
==============================================================================
# RELATÓRIO FINAL DE CONCLUSÃO
# ==============================================================================
print("\n" + "="*60)
print("🏁 VARREDURA DE GRADE FINALIZADA!")
print(f"🟢 Tarefas enviadas com sucesso: {sucessos}")
print(f"🟡 Setores ignorados (sem amostras/vazios): {pulados}")
print(f"🔴 Setores que geraram falha interna: {erros}")
print("="*60)
print("Dica: Você pode fechar o Colab se quiser. O processamento das imagens continuará")
print("rodando direto na nuvem do Google e os arquivos vão aparecer na sua pasta do Drive.")

SyntaxError: invalid syntax (4216324331.py, line 1)